# M1 Notebook 10 — Integration and Accumulation

**Notebook ID:** M1_N10  
**Status:** Runnable first edition  
**Random seed:** 42

> Derivatives describe instantaneous change. Integrals accumulate change across intervals.


## 1. Learning objectives

1. Interpret definite integrals as signed accumulation.
2. Approximate integrals with Riemann sums.
3. Apply midpoint, trapezoidal, and Simpson rules.
4. Understand cumulative integration.
5. Verify the Fundamental Theorem of Calculus numerically.
6. Estimate integrals through Monte Carlo simulation.
7. Connect integration to statistics, AI, and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.calculus import (
    cumulative_trapezoid,
    derivative,
    left_riemann,
    midpoint_rule,
    monte_carlo_integral,
    right_riemann,
    simpson_rule,
    trapezoidal_rule,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad

set_seed(42)
environment_info()


## 2. Definite integral

The definite integral

\[
\int_a^b f(x)\,dx
\]

represents signed accumulation over \([a,b]\).


## 3. Riemann sums

Partition \([a,b]\) into \(n\) subintervals of width

\[
\Delta x=\frac{b-a}{n}.
\]

A Riemann sum is

\[
\sum_{i=1}^{n}f(x_i^*)\Delta x.
\]


In [ ]:
f = lambda x: x**2
exact = 1/3

approximations = pd.DataFrame({
    "method": ["Left", "Right", "Midpoint", "Trapezoidal", "Simpson"],
    "estimate": [
        left_riemann(f, 0.0, 1.0, n=100),
        right_riemann(f, 0.0, 1.0, n=100),
        midpoint_rule(f, 0.0, 1.0, n=100),
        trapezoidal_rule(f, 0.0, 1.0, n=100),
        simpson_rule(f, 0.0, 1.0, n=100),
    ],
})
approximations["absolute_error"] = np.abs(approximations["estimate"] - exact)
approximations


## 4. Riemann-sum visualization

In [ ]:
a, b, n = 0.0, 1.0, 8
edges = np.linspace(a, b, n + 1)
width = (b - a) / n
left_x = edges[:-1]

x_plot = np.linspace(a, b, 400)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, f(x_plot))
ax.bar(left_x, f(left_x), width=width, align="edge", alpha=0.4, edgecolor="black")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Left Riemann Sum for x²")
plt.show()


For the increasing function \(x^2\) on \([0,1]\), the left sum underestimates and the right sum overestimates the integral.


In [ ]:
left = left_riemann(f, 0.0, 1.0, n=1000)
right = right_riemann(f, 0.0, 1.0, n=1000)

assert left < exact < right
left, exact, right


## 5. Convergence with increasing resolution

In [ ]:
n_values = np.array([10, 20, 50, 100, 200, 500, 1000])
method_errors = []

for n in n_values:
    method_errors.append({
        "n": n,
        "midpoint_error": abs(midpoint_rule(f, 0.0, 1.0, n=n) - exact),
        "trapezoid_error": abs(trapezoidal_rule(f, 0.0, 1.0, n=n) - exact),
        "simpson_error": abs(simpson_rule(f, 0.0, 1.0, n=n if n % 2 == 0 else n + 1) - exact),
    })

error_table = pd.DataFrame(method_errors)
error_table


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(error_table["n"], error_table["midpoint_error"], label="Midpoint")
ax.loglog(error_table["n"], error_table["trapezoid_error"], label="Trapezoidal")
ax.loglog(error_table["n"], error_table["simpson_error"], label="Simpson")
ax.set_xlabel("Number of subintervals")
ax.set_ylabel("Absolute error")
ax.set_title("Numerical Integration Convergence")
ax.legend()
plt.show()


## 6. Fundamental Theorem of Calculus

If

\[
F(x)=\int_a^x f(t)\,dt,
\]

then under suitable conditions,

\[
F'(x)=f(x).
\]


In [ ]:
x_grid = np.linspace(0.0, 3.0, 400)
y_grid = np.sin(x_grid)
F_grid = cumulative_trapezoid(y_grid, x_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_grid, y_grid, label="f(x)=sin(x)")
ax.plot(x_grid, F_grid, label="Accumulation F(x)")
ax.set_xlabel("x")
ax.set_title("Function and Cumulative Integral")
ax.legend()
plt.show()


In [ ]:
def accumulated_sine(x):
    grid = np.linspace(0.0, x, 2000)
    return trapezoidal_rule(np.sin, 0.0, x, n=2000)

point = 1.2
recovered = derivative(accumulated_sine, point, h=1e-4)
target = np.sin(point)

assert np.isclose(recovered, target, rtol=1e-4)
recovered, target


## 7. Comparison with SciPy quadrature

In [ ]:
scipy_value, scipy_error = quad(lambda x: np.exp(-x**2), 0.0, 1.0)
simpson_value = simpson_rule(lambda x: np.exp(-x**2), 0.0, 1.0, n=1000)

{
    "scipy_quad": scipy_value,
    "reported_quad_error": scipy_error,
    "simpson": simpson_value,
    "difference": abs(scipy_value - simpson_value),
}


## 8. Monte Carlo integration

For \(X\sim\operatorname{Uniform}(a,b)\),

\[
\int_a^b f(x)\,dx
=
(b-a)\mathbb E[f(X)].
\]


In [ ]:
sample_sizes = [100, 1_000, 10_000, 100_000]
mc_results = []

for samples in sample_sizes:
    estimate = monte_carlo_integral(
        f,
        0.0,
        1.0,
        samples=samples,
        seed=42,
    )
    mc_results.append({
        "samples": samples,
        "estimate": estimate,
        "absolute_error": abs(estimate - exact),
    })

pd.DataFrame(mc_results)


Monte Carlo convergence is slower in one dimension than high-order deterministic quadrature, but it becomes especially useful in high-dimensional integration.


## 9. Statistics interpretation

Integration is central to:

- probability normalization;
- expectations and moments;
- cumulative distribution functions;
- marginalization;
- Bayesian evidence;
- continuous likelihoods.


## 10. AI interpretation

Integration appears in:

- expected losses;
- probabilistic models;
- continuous-time neural systems;
- diffusion models;
- reinforcement-learning returns;
- uncertainty propagation;
- variational inference.


## 11. Decision Intelligence case — Accumulated resource deficit

Suppose daily demand exceeds supply during part of a planning horizon. The integral of the positive deficit estimates total unmet demand.


In [ ]:
days = np.linspace(0, 30, 601)

demand = 100 + 20*np.sin(2*np.pi*days/30)
supply = 105 + 8*np.cos(2*np.pi*days/30)
deficit = np.maximum(demand - supply, 0.0)

cumulative_deficit = cumulative_trapezoid(deficit, days)
total_deficit = cumulative_deficit[-1]

{
    "total_unmet_demand_units": total_deficit,
    "peak_daily_deficit": deficit.max(),
}


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(days, demand, label="Demand")
ax.plot(days, supply, label="Supply")
ax.fill_between(days, supply, demand, where=demand > supply, alpha=0.3, label="Deficit")
ax.set_xlabel("Day")
ax.set_ylabel("Resource units")
ax.set_title("Demand, Supply, and Accumulated Deficit")
ax.legend()
plt.show()


### Interpretation

The integral measures cumulative shortage under the modeled curves. A policy decision still requires uncertainty analysis, priority rules, cost constraints, distributional impacts, and operational feasibility.


## 12. Engineering notes

- Numerical integration error depends on smoothness and grid resolution.
- Simpson's rule requires an even number of subintervals.
- Irregularly spaced data require methods using the actual grid.
- Monte Carlo estimates should report uncertainty.
- Extrapolating beyond observed intervals can dominate accumulated totals.
- Signed area and total absolute area are different quantities.


## 13. Common errors

- Forgetting the width factor in a Riemann sum.
- Interpreting signed area as total physical magnitude.
- Applying Simpson's rule with an odd number of subintervals.
- Ignoring units after integration.
- Treating interpolation as observed reality.
- Integrating a biased model and trusting the accumulated result.


## 14. Exercises

### Level A
Explain integration as accumulation.

### Level B
Derive the trapezoidal rule from linear interpolation.

### Level C
Implement cumulative integration on irregularly spaced data.

### Capstone
Model a resource supply-demand gap, estimate cumulative shortage with several methods, quantify numerical error, and discuss the decision implications.


## 15. Key insight

Integration converts rates, densities, and local quantities into totals. It is the mathematics of accumulation, connecting calculus to probability, statistics, AI, simulation, and long-horizon Decision Intelligence.
